# 📚 Gerador de Dataset para Fine-Tuning (Instrução + Resposta)

Este notebook cria um dataset no formato `question` / `answer`, a partir do documento: Manual de Aplicação da NR 12 (249 páginas).

**Modelo de Linguagem:** Qwen2.5-1.5B-Instruct.

**max_chunk_length:** 300/700.
Podemos entender o Chunk como um conjunto de parágrafos ou textos com um limite de caracteres, quando definimos o valor do máximo de chunks, definimos esse limite. Pela lógica, pode-se imaginar que a variação de chunks nos possibilita resolver situações diferentes.
Para entendermos a diferença, utilizaremos com referencial o valor 500, acima dele teremos mais texto para uma chunk, ou seja, mais contexto para o modelo, contudo teremos menos quantidades de chunks, e pode ultrapassar o limite de tokens do modelo. Já para valores abaixo do referencial, teriámos uma chunk menor, e teóricamente mais precisa, e embora ela possa ficar com conceitos incompletos, dando respostas quebradas, teríamos maior número de pares. Assim, concluímos que ao utlizar o valor 300, teremos respostas mais curtas e precisas, e para o valor 700, conseguiríamos respostas mais densas e completas.

**Prompt:** Português / Pede respostas mais detalhadas (Mínimo de 30 e máximo de 100 palavras).

**max_new_tokens:** 150/300.
Controla a quantidade de tokens que pode ser gerado na resposta.
Quanto maior o limite de palavras do prompt e menor o da chunks, é interessante um valor mais alto, para evitar cortes ou quebras na resposta.

**Saída:** arquivo JSONL (uma linha por exemplo), pronto para treinar ou ajustar um modelo RAG ou de QA.

---
## Etapas principais:
1. Instalar dependências;
2. Extrair texto do arquivo PDF;
3. Dividir o texto em Chunks;
4. Para cada trecho, usar um modelo Hugging Face para gerar uma pergunta e uma resposta curta;
5. Salvar os pares gerados em formato JSONL;

## 1. Instalação das bibliotecas

In [1]:
pip install transformers torch accelerate pdfplumber tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 85.2 MB/s eta 0:00:00


In [2]:
#pip install --upgrade pip

## 2. Importações e configurações

In [3]:
import pdfplumber
import json
import re
import torch
from transformers import pipeline, logging
from tqdm import tqdm

# Desativa avisos (apenas erros críticos serão mostrados)
logging.set_verbosity_error()

## 3. Função para extrair texto

A extração começa da página 12 e vai até a página 246, evitando capa, sumário e referências.
- **PDF** → `pdfplumber`


In [ ]:
def extract_text_from_file(file_path):
    """
    Extrai texto do arquivo Manual NR12.
    Para PDFs, lê apenas das páginas 12 até 246.
    Evitando capa, sumário e referências.
    Retorna uma string com todo o conteúdo.
    """
    if file_path.lower().endswith('.pdf'):
        text = ""

        with pdfplumber.open(file_path) as pdf:
            # pdf.pages usa índice começando em 0
            start_page = 11  # página 12
            end_page = 246   # fim exclusivo (inclui a página 246)

            for page in pdf.pages[start_page:end_page]:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        return text

    elif file_path.lower().endswith('.txt'):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()

    else:
        raise ValueError("Formato não suportado. Use .pdf ou .txt")


## 4. Divisão em trechos (chunks)

Modelos de linguagem têm um limite de tokens. Por isso dividimos o texto em blocos. Cada bloco será usado para gerar um par pergunta/resposta.
Como estamos tratando de um texto mais técnico, o objetivo é que a resposta seja mais longa e completa, por isso utlizamos `max_chunck_length=700`

In [5]:
def split_text(text, max_chunk_length=700):
    """
    Divide o texto em chunks com base em quebras de linha.
    Cada chunk terá no máximo max_chunk_length caracteres.
    """
    paragraphs = text.split("\n")
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_length:
            current_chunk += para + "\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


## 5. Gerar (Instrução, Resposta) a partir de um trecho

Usamos um modelo da família **Qwen2.3.5B-Instruct**. O modelo recebe um prompt bem estruturado e específico para a norma regulamentadora 12 e deve retornar algo como:

```
INSTRUCTION: Como as máquinas devem ser dimensionadas de acordo com a NR-12?"
RESPONSE: As máquinas devem ser adequadamente dimensionadas em conformidade com as normas técnicas oficiais ou as normas internacionais aplicáveis, conforme estabelecido na NR-12.
```

In [6]:
def generate_instruction_response(chunk, hf_pipeline):

    prompt = f"""
Sua tarefa é analisar o conteúdo do Manual NR-12 e gerar UM ÚNICO par de PERGUNTA e RESPOSTA para fine-tuning de modelo de linguagem.

REGRAS OBRIGATÓRIAS:

1. Utilize exclusivamente o conteúdo técnico, normativo e explicativo presente no corpo do documento.
2. Ignore completamente:

   - Sumário;
   - Índices;
   - Capa;
   - Dados de autoria;
   - Créditos;
   - Referências bibliográficas;
   - Rodapés administrativos;
   - Anexos sem conteúdo normativo;
   - Legendas de imagens;
   - Tabelas de conteúdo;
   - Numeração de páginas;
   - Imagens e elementos visuais;

3. Concentre-se apenas nas informações que transmitam conhecimento, definições, requisitos, obrigações, procedimentos, medidas de segurança, conceitos, responsabilidades, exceções e critérios técnicos.
4. Gere uma pergunta clara, objetiva e diretamente respondida pelo texto.
5. A resposta deve ser COMPLETA, DETALHADA e EXPLICATIVA: escreva um parágrafo corrido que contextualize a informação (não apenas uma definição curta de uma linha), sempre baseada exclusivamente no conteúdo do documento.
6. Não invente informações que não estejam explicitamente presentes no texto.
7. Quando possível, diversifique os tipos de perguntas:

   - Definições;
   - Requisitos obrigatórios;
   - Procedimentos;
   - Responsabilidades;
   - Medidas de proteção;
   - Conceitos técnicos;
   - Exceções e restrições;
   - Critérios de conformidade.

8. O par deve ser independente e compreensível sem necessidade de contexto adicional.
9. Não inclua comentários, explicações, títulos ou qualquer texto fora do formato especificado.
10. A pergunta (INSTRUCTION) deve ter no máximo 40 palavras.
11. A resposta (RESPONSE) deve ter no MÍNIMO 30 palavras e no máximo 100 palavras.
12. Gere APENAS UM par INSTRUCTION/RESPONSE. Assim que terminar de escrever a RESPONSE, PARE IMEDIATAMENTE. Não inicie um novo "INSTRUCTION:", não gere pares adicionais e não escreva nada após a resposta.

FORMATO OBRIGATÓRIO DE SAÍDA (gere exatamente isso, e nada mais):

INSTRUCTION: <pergunta>
RESPONSE: <resposta>

Conteúdo:
\"\"\"
{chunk}
\"\"\"
"""

    messages = [{"role": "user", "content": prompt}]

    try:
        outputs = hf_pipeline(
            messages,
            max_new_tokens=350,
            max_length=None,  # evita warning
            return_full_text=False
        )
        content = outputs[0]["generated_text"]

        # CORREÇÃO: o modelo às vezes "vaza" e, depois de terminar a
        # RESPONSE, começa a gerar um novo par (ex.: "...\n\nINSTRUCTION: ...").
        # A regex abaixo captura apenas o PRIMEIRO par INSTRUCTION/RESPONSE e
        # corta a RESPONSE assim que encontra um próximo "INSTRUCTION:" (ou o
        # fim do texto), evitando que esse trecho extra fique misturado
        # dentro do campo "Output".
        match = re.search(
            r"INSTRUCTION:\s*(.*?)\s*RESPONSE:\s*(.*?)(?=\n*INSTRUCTION:|\Z)",
            content,
            re.DOTALL,
        )

        if not match:
            # Saída fora do formato esperado: descarta o par
            return None, None

        instr_part = match.group(1).strip()
        answer = match.group(2).strip()

        if not instr_part or not answer:
            return None, None

        return instr_part, answer
    except Exception as e:
        # Em caso de erro, retorna None, None (o par será ignorado)
        return None, None


## 6. Salvar dataset em formato JSONL

Cada linha do arquivo será um objeto `{"question": "...", "answer": "..."}`. Esse formato é amplamente usado para fine-tuning de modelos de QA.

In [7]:
def save_to_jsonl(pairs, output_file="dataset.jsonl"):
    """
    pairs: lista de tuplas (instrução, resposta)
    output_file: nome do arquivo de saída
    """
    with open(output_file, "w", encoding="utf-8") as f:
        for instruction, answer in pairs:
            if instruction and answer:
                example = {
                    "Instruction": instruction,
                    "Output": answer
                }
                f.write(json.dumps(example, ensure_ascii=False) + "\n")


## ⚙️ Diagnóstico de hardware (recomendado antes de carregar o modelo)

Execute esta célula para verificar RAM disponível e presença de GPU antes de carregar o modelo.
Isso evita crashes silenciosos por falta de memória.

In [ ]:
def generate_dataset(file_path, model_id="Qwen/Qwen2.5-3B-Instruct",
                    output_file="dataset.jsonl", max_chunks=None):
    """
    file_path: caminho para o arquivo .pdf
    model_id: identificador do modelo no Hugging Face Hub
    output_file: nome do arquivo de saída (JSONL)
    max_chunks: se especificado, limita a quantidade de chunks processados
    """
    print(f"🔄 Carregando modelo: {model_id} ...")

    # Pipeline para geração de texto
    # SE NECESSÁRIO: device=-1 força CPU (evita crash com GPU integrada AMD sem suporte ROCm/CUDA)
    # SE NECESSÁRIO: torch.float32 no lugar de bfloat16 (compatível com CPUs sem AVX-512)
    hf_pipeline = pipeline(
        "text-generation",
        model=model_id,
        device_map="auto",        # usa GPU se disponível
        torch_dtype=torch.bfloat16  # bfloat16 causa crash em CPUs sem suporte nativo
    )

    print("📄 Extraindo texto do arquivo...")
    text = extract_text_from_file(file_path)

    print("✂️  Dividindo em chunks...")
    chunks = split_text(text)

    if max_chunks:
        chunks = chunks[:max_chunks]

    print(f"🧠 Gerando pares (instrução + resposta) para {len(chunks)} chunks...")
    pairs = []

    for chunk in tqdm(chunks, desc="Processando chunks"):
        if len(chunk.strip()) < 10:  # ignora chunks muito curtos
            continue
        instruction, answer = generate_instruction_response(chunk, hf_pipeline)
        if instruction and answer:
            pairs.append((instruction, answer))

    save_to_jsonl(pairs, output_file)
    print(f"\n✅ Dataset salvo em: {output_file} ({len(pairs)} exemplos gerados)")


## 7. Função principal

A função `generate_dataset` coordena todo o fluxo:
- Carrega o modelo (baixa os pesos se necessário)
- Extrai texto do arquivo (PDF)
- Divide em chunks
- Para cada chunk, chama o modelo e coleta os pares
- Salva o resultado

## 8. Execução

Agora basta informar o caminho do seu arquivo (PDF ou TXT) e, opcionalmente, o número máximo de chunks para testar.

> **#** começar com `max_chunks=10` para verificar se a geração está funcionando. Depois remover o parâmetro ou aumentar o valor para processar o documento inteiro.

O modelo será baixado na primeira execução. E NECESSÁRIO AGUARDAR O DOWNLOAD.

In [ ]:
# Exemplo de uso com um arquivo .pdf
caminho_arquivo = "Manual_NR12.pdf"   # Altere para o seu arquivo
# caminho_arquivo = "meu_texto.txt"      # Ou use um .txt

generate_dataset(
    file_path=caminho_arquivo,
    model_id="Qwen/Qwen2.5-3B-Instruct",
    output_file="dataset.jsonl",
    #max_chunks=10   # remova esta linha para processar todos os chunks
)

## 🔍 Observações finais

- Com 700 Chuncks e o prompt correto, o modelo `Qwen/Qwen2.5-3B-Instruct` conseguiu criar mais de 400 pares com o Manual_NR12
- Foi registrado poucos erros no Dataset gerado.